# Teste das codificações quânticas

Este notebook permite digitar um vetor de features, aplicar `angle_encoding` e `amplitude_encoding` de `src/codification.py` e inspecionar o estado quântico correspondente em um simulador simples.

A simulação abaixo não depende de PennyLane ou Qiskit. Ela calcula diretamente o vetor de estado para verificar se a preparação está coerente.

In [1]:
import math
from pathlib import Path

import torch

from src.codification import angle_encoding, amplitude_encoding

torch.set_printoptions(precision=6, sci_mode=False)

## 1. Defina o vetor e o número de qubits

Altere `features` e `num_qubits` livremente.

Lembrete:

- `angle_encoding` prepara `num_qubits` valores;
- `amplitude_encoding` prepara `2 ** num_qubits` amplitudes.

In [7]:
features = torch.tensor([0.2, -1.5, 3.0, 0.7, 2.1, 0.5, 5.3, 2.1], dtype=torch.float32)
num_qubits = 3

features, num_qubits

(tensor([ 0.200000, -1.500000,  3.000000,  0.700000,  2.100000,  0.500000,
          5.300000,  2.100000]),
 3)

## 2. Funções auxiliares para simular estado quântico

Para `angle_encoding`, vamos assumir que cada ângulo é aplicado com uma porta `RY(theta)` em um qubit iniciado em `|0>`.

Para `amplitude_encoding`, o estado quântico já é diretamente o vetor normalizado retornado pela função.

In [8]:
def kron_all(states):
    result = states[0]
    for state in states[1:]:
        result = torch.kron(result, state)
    return result


def state_from_ry_angles(angles):
    single_qubit_states = []
    for theta in angles:
        single_qubit_states.append(
            torch.stack([torch.cos(theta / 2), torch.sin(theta / 2)]).to(torch.float32)
        )
    return kron_all(single_qubit_states)


def basis_labels(num_qubits):
    return [format(index, f"0{num_qubits}b") for index in range(2 ** num_qubits)]


def print_state(state, num_qubits, threshold=1e-8):
    labels = basis_labels(num_qubits)
    for label, amplitude in zip(labels, state):
        if abs(float(amplitude)) > threshold:
            probability = float(amplitude ** 2)
            print(f"|{label}>: amplitude={float(amplitude): .6f}, probability={probability: .6f}")
    print(f"\nNorma L2 do estado: {float(torch.linalg.vector_norm(state)): .6f}")

## 3. Angle encoding

A função prepara `num_qubits` ângulos. Aqui simulamos o estado resultante de aplicar `RY(theta_i)` em cada qubit.

In [9]:
angles = angle_encoding(features, num_qubits=num_qubits)
angle_state = state_from_ry_angles(angles)

print("Features originais:")
print(features)
print("\nÂngulos preparados para RY:")
print(angles)
print("\nEstado após angle encoding com RY em cada qubit:")
print_state(angle_state, num_qubits)

Features originais:
tensor([ 0.200000, -1.500000,  3.000000,  0.700000,  2.100000,  0.500000,
         5.300000,  2.100000])

Ângulos preparados para RY:
tensor([-0.767945, -3.141593,  3.141593])

Estado após angle encoding com RY em cada qubit:
|001>: amplitude=-0.000000, probability= 0.000000
|010>: amplitude= 0.000000, probability= 0.000000
|011>: amplitude=-0.927184, probability= 0.859670
|101>: amplitude= 0.000000, probability= 0.000000
|110>: amplitude=-0.000000, probability= 0.000000
|111>: amplitude= 0.374607, probability= 0.140330

Norma L2 do estado:  1.000000


## 4. Amplitude encoding

A função prepara um vetor de tamanho `2 ** num_qubits` e normaliza para norma 1. Esse vetor já é o estado quântico no espaço computacional.

In [10]:
amplitude_state = amplitude_encoding(features, num_qubits=num_qubits)

print("Features originais:")
print(features)
print(f"\nVetor preparado para amplitude encoding com dimensão 2^{num_qubits} = {2 ** num_qubits}:")
print(amplitude_state)
print("\nEstado após amplitude encoding:")
print_state(amplitude_state, num_qubits)

Features originais:
tensor([ 0.200000, -1.500000,  3.000000,  0.700000,  2.100000,  0.500000,
         5.300000,  2.100000])

Vetor preparado para amplitude encoding com dimensão 2^3 = 8:
tensor([ 0.028589, -0.214417,  0.428834,  0.100061,  0.300184,  0.071472,
         0.757607,  0.300184])

Estado após amplitude encoding:
|000>: amplitude= 0.028589, probability= 0.000817
|001>: amplitude=-0.214417, probability= 0.045975
|010>: amplitude= 0.428834, probability= 0.183899
|011>: amplitude= 0.100061, probability= 0.010012
|100>: amplitude= 0.300184, probability= 0.090110
|101>: amplitude= 0.071472, probability= 0.005108
|110>: amplitude= 0.757607, probability= 0.573968
|111>: amplitude= 0.300184, probability= 0.090110

Norma L2 do estado:  1.000000


## 5. Comparação rápida das dimensões

Esta célula resume o tamanho usado por cada codificação para o mesmo número de qubits.

In [6]:
print(f"Número de qubits: {num_qubits}")
print(f"Dimensão usada no angle encoding: {angles.numel()}")
print(f"Dimensão usada no amplitude encoding: {amplitude_state.numel()}")
print(f"Dimensão do espaço de estados: {2 ** num_qubits}")

Número de qubits: 3
Dimensão usada no angle encoding: 3
Dimensão usada no amplitude encoding: 8
Dimensão do espaço de estados: 8


## Observação

Este notebook verifica apenas a preparação do estado. O próximo passo do projeto será inserir esses estados em um circuito variacional parametrizado, usando uma biblioteca como PennyLane ou Qiskit.